# Gemma 4 E4B Fine-Tuning for Hable Ya

Fine-tune Gemma 4 E4B on SFT and DPO datasets generated from eval fixtures.

**Prerequisites:**
- Generate datasets: `python -m finetune.generate`
- Download HF weights: `python scripts/download_model.py --hf-only`
- GPU with >= 16 GB VRAM (Unsloth 4-bit)

**Outputs:**
- LoRA adapter saved to `models/gemma-4-e4b-lora/`
- Merged GGUF exported to `models/gemma-4-e4b-finetuned.gguf`

In [1]:
# Install dependencies (run once)
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install datasets trl

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

HF_TOKEN = os.environ["HF_TOKEN"]

In [3]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # assumes running from notebooks/
DATASETS_DIR = REPO_ROOT / "finetune" / "datasets"
MODELS_DIR = REPO_ROOT / "models"
OUTPUT_DIR = MODELS_DIR / "gemma-4-e4b-lora"

MODEL_NAME = "unsloth/gemma-4-E4B-it"
LOCAL_WEIGHTS = MODELS_DIR / "gemma-4-e4b-hf"

# Use local weights if already downloaded, otherwise pull from HF
MODEL_PATH = str(LOCAL_WEIGHTS) if LOCAL_WEIGHTS.exists() else MODEL_NAME
print(f"Model source: {MODEL_PATH}")

Model source: /home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-hf


## 1. Load Model with Unsloth

In [4]:
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/adrian/Desarrollador/hable-ya/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.447 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2130/2130 [00:05<00:00, 397.25it/s]


In [5]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 16,           # Larger = higher accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)
model.print_trainable_parameters()

trainable params: 36,700,160 || all params: 8,032,856,608 || trainable%: 0.4569


## 2. Load SFT Dataset

In [6]:
import json

def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

sft_records = load_jsonl(DATASETS_DIR / "sft_train.jsonl")
print(f"Loaded {len(sft_records)} SFT examples")
print(f"Keys: {list(sft_records[0].keys())}")
print(f"Roles: {[m['role'] for m in sft_records[0]['messages']]}")

Loaded 1181 SFT examples
Keys: ['messages', 'metadata']
Roles: ['system', 'assistant', 'user', 'assistant', 'user', 'assistant']


In [7]:
from collections import Counter

bands = Counter(r["metadata"]["cefr_band"] for r in sft_records)
categories = Counter(r["metadata"]["category"] for r in sft_records)

print("CEFR distribution:")
for band in ["A1", "A2", "B1", "B2", "C1"]:
    print(f"  {band}: {bands.get(band, 0)}")

print("\nCategory distribution:")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

CEFR distribution:
  A1: 261
  A2: 241
  B1: 220
  B2: 211
  C1: 248

Category distribution:
  multi_error: 318
  single_error_recast: 676
  tool_call_correctness: 187


In [8]:
from datasets import Dataset

def format_for_training(record: dict) -> dict:
    """Apply the chat template to convert messages into a single text field."""
    text = tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

sft_dataset = Dataset.from_list(sft_records).map(format_for_training)
print(f"Dataset size: {len(sft_dataset)}")
print(f"\nSample (first 500 chars):\n{sft_dataset[0]['text'][:500]}")

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1181/1181 [00:00<00:00, 3509.34 examples/s]

Dataset size: 1181

Sample (first 500 chars):
<bos><|turn>system
You are a Spanish conversation partner for a language learner.

## Learner
- CEFR level: A1
- L1 reliance: 0.40 (higher = more likely to fall back on English)
- Speech fluency: 0.30
- Known error patterns: ser_estar, gender_agreement
- Vocabulary strengths: greetings, numbers, colors

## Topic: city_life
Habla con el estudiante sobre su ciudad y los lugares que visita.
Target structures: estar + location, hay, basic prepositions.

## Response format (strict)
- 1 to 3 sentences


## 3. SFT Training

In [9]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    packing=False,
    args=SFTConfig(
      per_device_train_batch_size = 2,       # if memory allows; helps diversity per step
      gradient_accumulation_steps = 4,        # effective batch = 8
      warmup_ratio = 0.03,                    # warmup_steps=5 is too abrupt at this LR
      num_train_epochs = 2,
      learning_rate = 5e-5,                   # was 2e-4 — an order of magnitude down
      logging_steps = 10,
      save_strategy = "epoch",                # checkpoint each epoch so you can A/B
      optim = "adamw_8bit",
      weight_decay = 0.01,                    # was 0.001 — light regularization
      lr_scheduler_type = "cosine",           # gentler than linear at the end
      seed = 3407,
      report_to = "none",
  ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Tokenizing ["text"] (num_proc=20): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1181/1181 [00:42<00:00, 27.95 examples/s]


In [10]:
from unsloth.chat_templates import train_on_responses_only
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Filter (num_proc=20): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1181/1181 [00:00<00:00, 1459.67 examples/s]


In [11]:
sft_stats = sft_trainer.train()
print(f"\nTraining loss: {sft_stats.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,181 | Num Epochs = 2 | Total steps = 296
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,13.929088
20,6.908216
30,4.079131
40,2.943591
50,2.133501
60,1.877218
70,1.628705
80,1.350637
90,1.118675
100,1.052511



Training loss: 1.7347


## 4. Quick Inference Check

Sanity-check the fine-tuned model before continuing to DPO.

In [15]:









FastModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": [{
            "type": "text",
            "text": "You are a Spanish conversation partner for language learners.\nThe learner is at CEFR level A2 (production_level=0.35).\nWhen the learner makes an error, recast it naturally in your response -- never correct explicitly. Keep responses to 1-3 sentences with exactly one question. After responding, call log_turn with the learner utterance, errors, and fluency indicators."
        }],
    },
    {"role": "assistant", "content": [{"type": "text", "text": "Hola, \u00bfqu\u00e9 hiciste hoy?"}]},
    {"role": "user", "content": [{"type": "text", "text": "Yo fui al tienda y compr\u00e9 mucho frutas."}]},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.9,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("Model response:\n")
print(response)

Model response:

¡Qué bien! Compraste muchas frutas frescas. ¿Qué frutas elegiste?

log_turn({
    'learner_utterance': 'Yo fui al tienda y compré mucho frutas.',
    'errors': ['al tienda', 'mucho frutas'],
    'fluency': {
        'score': 0.7,
        'breakdown': {
            'pauses': 0,
            'repetitions': 0,
            'filler_words': 0
        }
    }
})


## 6. Save and Export

Save the final LoRA adapter and export a merged GGUF for llama.cpp deployment.

In [16]:
# Save final adapter
final_adapter_path = OUTPUT_DIR / "final-adapter"
model.save_pretrained(str(final_adapter_path))
tokenizer.save_pretrained(str(final_adapter_path))
print(f"Final adapter saved to {final_adapter_path}")

Final adapter saved to /home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-lora/final-adapter


In [17]:
model.save_pretrained_gguf(
        MODELS_DIR / "gemma-4-e4b-finetuned",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-hf
Found HuggingFace hub cache directory: /home/adrian/.cache/huggingface/hub


Unsloth: Merging weights into 16bit: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:54<00:00, 54.85s/it]


Unsloth: Merge process complete. Saved to `/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned_gguf/gemma-4-e4b-hf.BF16.gguf', '/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned_gguf/gemma-4-e4b-hf.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth

{'save_directory': PosixPath('/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned'),
 'gguf_directory': '/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned_gguf',
 'gguf_files': ['/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned_gguf/gemma-4-e4b-hf.Q8_0.gguf',
  '/home/adrian/Desarrollador/hable-ya/models/gemma-4-e4b-finetuned_gguf/gemma-4-e4b-hf.BF16-mmproj.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': True,
 'fix_bos_token': False}

In [ ]:
# Upload to HF
model.push_to_hub_merged(
    "adrianmoses/gemma-4-e4b-hable-ya",
    tokenizer,
    token = HF_TOKEN
)

In [ ]:
model.push_to_hub_gguf(
    "adrianmoses/gemma-4-e4b-hable-ya-gguf",
    tokenizer,
    quantization_method = "Q8_0",
    token = HF_TOKEN
)